# Ejercicio 01 · Esquema y seeds de `monitoreo`

Este cuaderno reemplaza a Oracle: crea la base **SQLite** `db.sqlite3` en la raíz del proyecto,
define las 5 tablas que usan los modelos de Django (`managed = False`) y carga datos de prueba.

**Flujo:** Markdown (explicación) → Python (conexión / análisis) → SQL (esquema, seeds, consultas).

> Ejecuta las celdas en orden (*Run All*). Puedes re-ejecutarlo cuando quieras: reinicia las tablas desde cero.

## Paso 1 · Conexión (Python)
Ubicamos `db.sqlite3` junto a `manage.py` y activamos la extensión `%sql` de JupySQL.

In [1]:
from pathlib import Path
from sqlalchemy import create_engine, event
import pandas as pd

# El notebook vive en notebooks/, la base va en la raíz del proyecto
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = BASE_DIR / "db.sqlite3"

engine = create_engine(f"sqlite:///{DB_PATH}")

@event.listens_for(engine, "connect")
def _activar_fk(dbapi_conn, _):
    dbapi_conn.execute("PRAGMA foreign_keys = ON")

%load_ext sql
%config SqlMagic.displaylimit = 50
%sql engine
print("Base de datos:", DB_PATH)

Base de datos: E:\saoto\Documents\Portafolio\producto-digital-alta-escalabilidad\db.sqlite3


## Paso 2 · Esquema (SQL)
Las tablas replican `monitoreo/models.py`. Django nombra las llaves foráneas como `<campo>_id`
(por ejemplo `marca` → `marca_id`).

```
MARCAS ─< MODELOS ─< VEHICULOS ─< PRECIOS_VEHICULOS
                        └── ESTADOS_VEHICULO
```

In [2]:
%%sql
DROP TABLE IF EXISTS PRECIOS_VEHICULOS;
DROP TABLE IF EXISTS VEHICULOS;
DROP TABLE IF EXISTS ESTADOS_VEHICULO;
DROP TABLE IF EXISTS MODELOS;
DROP TABLE IF EXISTS MARCAS;

CREATE TABLE MARCAS (
    id              INTEGER PRIMARY KEY,
    nombre          VARCHAR(100) NOT NULL UNIQUE,
    pais_origen     VARCHAR(50),
    fecha_creacion  DATETIME NOT NULL DEFAULT CURRENT_TIMESTAMP,
    activo          INTEGER  NOT NULL DEFAULT 1
);

CREATE TABLE MODELOS (
    id               INTEGER PRIMARY KEY,
    marca_id         INTEGER NOT NULL REFERENCES MARCAS(id),
    nombre           VARCHAR(100) NOT NULL,
    anio_inicio      INTEGER NOT NULL,
    tipo_carroceria  VARCHAR(20) NOT NULL
);

CREATE TABLE ESTADOS_VEHICULO (
    id                   INTEGER PRIMARY KEY,
    nombre               VARCHAR(50) NOT NULL UNIQUE,
    factor_depreciacion  DECIMAL(4,3) NOT NULL
);

CREATE TABLE VEHICULOS (
    id             INTEGER PRIMARY KEY,
    modelo_id      INTEGER NOT NULL REFERENCES MODELOS(id),
    anio_vehiculo  INTEGER NOT NULL,
    kilometraje    INTEGER NOT NULL,
    estado_id      INTEGER NOT NULL REFERENCES ESTADOS_VEHICULO(id),
    color          VARCHAR(30) NOT NULL
);

CREATE TABLE PRECIOS_VEHICULOS (
    id            INTEGER PRIMARY KEY,
    vehiculo_id   INTEGER NOT NULL REFERENCES VEHICULOS(id),
    precio        DECIMAL(12,2) NOT NULL,
    tipo_precio   VARCHAR(20) NOT NULL,
    fuente        VARCHAR(20) NOT NULL,
    fecha_precio  DATETIME NOT NULL
);

Running query in 'sqlite:///E%3A%5Csaoto%5CDocuments%5CPortafolio%5Cproducto-digital-alta-escalabilidad%5Cdb.sqlite3'

++
||
++
++

## Paso 3 · Seeds (SQL)
Datos de prueba controlados para los ejercicios. Modifícalos libremente y vuelve a ejecutar desde el Paso 2.

In [3]:
%%sql
INSERT INTO MARCAS (id, nombre, pais_origen, fecha_creacion, activo) VALUES
    (1, 'Toyota',    'Japón',          '2026-01-10 09:00:00', 1),
    (2, 'Chevrolet', 'Estados Unidos', '2026-01-10 09:00:00', 1),
    (3, 'Renault',   'Francia',        '2026-01-10 09:00:00', 1);

INSERT INTO MODELOS (id, marca_id, nombre, anio_inicio, tipo_carroceria) VALUES
    (1, 1, 'Corolla', 1966, 'SEDAN'),
    (2, 1, 'Hilux',   1968, 'PICKUP'),
    (3, 2, 'Onix',    2012, 'HATCHBACK'),
    (4, 3, 'Duster',  2010, 'SUV');

INSERT INTO ESTADOS_VEHICULO (id, nombre, factor_depreciacion) VALUES
    (1, 'Nuevo',           1.000),
    (2, 'Usado excelente', 0.850),
    (3, 'Usado regular',   0.650);

INSERT INTO VEHICULOS (id, modelo_id, anio_vehiculo, kilometraje, estado_id, color) VALUES
    (1, 1, 2026,      0, 1, 'Blanco'),
    (2, 1, 2021,  48000, 2, 'Gris'),
    (3, 2, 2019,  95000, 3, 'Negro'),
    (4, 3, 2024,  12000, 2, 'Rojo'),
    (5, 4, 2022,  61000, 3, 'Azul');

INSERT INTO PRECIOS_VEHICULOS (id, vehiculo_id, precio, tipo_precio, fuente, fecha_precio) VALUES
    (1, 1, 125000000.00, 'LISTA',   'CONCESIONARIO', '2026-09-01 10:00:00'),
    (2, 2,  82000000.00, 'MERCADO', 'CLASIFICADOS',  '2026-09-01 10:00:00'),
    (3, 2,  79500000.00, 'MERCADO', 'CLASIFICADOS',  '2026-09-15 10:00:00'),
    (4, 3, 118000000.00, 'MERCADO', 'SUBASTA',       '2026-09-05 10:00:00'),
    (5, 4,  61000000.00, 'LISTA',   'CONCESIONARIO', '2026-09-02 10:00:00'),
    (6, 4,  58900000.00, 'MERCADO', 'CLASIFICADOS',  '2026-09-20 10:00:00'),
    (7, 5,  64000000.00, 'MERCADO', 'SUBASTA',       '2026-09-10 10:00:00');

Running query in 'sqlite:///E%3A%5Csaoto%5CDocuments%5CPortafolio%5Cproducto-digital-alta-escalabilidad%5Cdb.sqlite3'

3 rows affected.

4 rows affected.

3 rows affected.

5 rows affected.

7 rows affected.

++
||
++
++

## Paso 4 · Verificación (SQL)
Conteo de filas por tabla para confirmar que los seeds cargaron.

In [4]:
%%sql
SELECT 'MARCAS' AS tabla, COUNT(*) AS filas FROM MARCAS
UNION ALL SELECT 'MODELOS', COUNT(*) FROM MODELOS
UNION ALL SELECT 'ESTADOS_VEHICULO', COUNT(*) FROM ESTADOS_VEHICULO
UNION ALL SELECT 'VEHICULOS', COUNT(*) FROM VEHICULOS
UNION ALL SELECT 'PRECIOS_VEHICULOS', COUNT(*) FROM PRECIOS_VEHICULOS;

Running query in 'sqlite:///E%3A%5Csaoto%5CDocuments%5CPortafolio%5Cproducto-digital-alta-escalabilidad%5Cdb.sqlite3'

tabla,filas
MARCAS,3
MODELOS,4
ESTADOS_VEHICULO,3
VEHICULOS,5
PRECIOS_VEHICULOS,7


## Paso 5 · Consulta de negocio (SQL)
Último precio registrado de cada vehículo con su marca, modelo y estado.

In [5]:
%%sql
SELECT ma.nombre AS marca, mo.nombre AS modelo, v.anio_vehiculo, v.kilometraje,
       e.nombre AS estado, p.precio, p.fuente, p.fecha_precio
FROM PRECIOS_VEHICULOS p
JOIN VEHICULOS v         ON v.id  = p.vehiculo_id
JOIN MODELOS mo          ON mo.id = v.modelo_id
JOIN MARCAS ma           ON ma.id = mo.marca_id
JOIN ESTADOS_VEHICULO e  ON e.id  = v.estado_id
WHERE p.fecha_precio = (SELECT MAX(fecha_precio) FROM PRECIOS_VEHICULOS WHERE vehiculo_id = p.vehiculo_id)
ORDER BY ma.nombre, mo.nombre;

Running query in 'sqlite:///E%3A%5Csaoto%5CDocuments%5CPortafolio%5Cproducto-digital-alta-escalabilidad%5Cdb.sqlite3'

marca,modelo,anio_vehiculo,kilometraje,estado,precio,fuente,fecha_precio
Chevrolet,Onix,2024,12000,Usado excelente,58900000,CLASIFICADOS,2026-09-20 10:00:00
Renault,Duster,2022,61000,Usado regular,64000000,SUBASTA,2026-09-10 10:00:00
Toyota,Corolla,2026,0,Nuevo,125000000,CONCESIONARIO,2026-09-01 10:00:00
Toyota,Corolla,2021,48000,Usado excelente,79500000,CLASIFICADOS,2026-09-15 10:00:00
Toyota,Hilux,2019,95000,Usado regular,118000000,SUBASTA,2026-09-05 10:00:00


## Paso 6 · Análisis (Python)
Llevamos los datos a pandas para calcular la variación de precio de cada vehículo.

In [6]:
historial = pd.read_sql(
    """SELECT vehiculo_id, precio, fecha_precio FROM PRECIOS_VEHICULOS ORDER BY vehiculo_id, fecha_precio""",
    engine,
)
variacion = (
    historial.groupby("vehiculo_id")["precio"]
    .agg(primer_precio="first", ultimo_precio="last", registros="count")
    .assign(variacion_pct=lambda d: ((d.ultimo_precio / d.primer_precio - 1) * 100).round(2))
)
variacion

,primer_precio,ultimo_precio,registros,variacion_pct
vehiculo_id,,,,
1,125000000,125000000,1,0.00
2,82000000,79500000,2,-3.05
3,118000000,118000000,1,0.00
4,61000000,58900000,2,-3.44
5,64000000,64000000,1,0.00


## Siguiente paso · Ver los datos en Django (Chrome)
Desde la raíz del proyecto:

```
.\venv\Scripts\python.exe manage.py migrate
.\venv\Scripts\python.exe manage.py createsuperuser
.\venv\Scripts\python.exe manage.py runserver
```

Abre <http://127.0.0.1:8000/admin/> — los modelos de **Monitoreo** muestran estos mismos seeds.